In [0]:
import builtins
from pyspark.sql import functions as F
from pyspark.sql.window import Window

B = "workspace.bronze"
S = "workspace.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")

def dedup(df, chave):
    """Mantém 1 linha por chave: a mais recente pela ingestion_datetime.
    Como current_timestamp() é igual para todas as linhas de uma mesma carga, há empate;
    desempata pela versão mais completa (mais colunas preenchidas após a limpeza)."""
    cols = [c for c in df.columns if c not in (chave, "ingestion_datetime")]
    # builtins.sum: no Databricks, "sum" puro é o F.sum do Spark (agregação), não o do Python
    completude = builtins.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in cols)
    w = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc(), completude.desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def salvar(df, nome):
    df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{S}.{nome}")
    print(f"OK: {nome} ({spark.table(f'{S}.{nome}').count()} linhas)")

In [0]:
df_info = spark.table(f"{B}.tb_movies_info").select(
    F.trim("id").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    F.col("release_date").alias("data_lancamento"),
    F.col("runtime").alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    F.col("status").alias("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
    "ingestion_datetime",
)

# Status: normaliza ANTES de traduzir (hífen/underscore -> espaço, espaços duplos, caixa baixa).
# "In-Production", "POST PRODUCTION" e "post production" viram o mesmo termo.
# Corrompidos (ex.: uma data caiu na coluna de status) ou não mapeáveis -> "Não Informado".
status_norm = F.lower(F.trim(F.regexp_replace(F.regexp_replace("status_filme", r"[-_]", " "), r"\s+", " ")))
df_info = df_info.withColumn("status_filme",
    F.when(status_norm == "released", "Lançado")
     .when(status_norm == "post production", "Pós-Produção")
     .when(status_norm == "in production", "Em Produção")
     .when(status_norm == "planned", "Planejado")
     .when(status_norm == "rumored", "Rumores")
     .when(status_norm.isin("canceled", "cancelled"), "Cancelado")
     .otherwise("Não Informado"))

# Data multi-formato (padrões reais da origem): yyyy-MM-dd, MM-dd-yyyy (com hífen) e dd/MM/yyyy (com barra).
# Verificado nos dados: com barra o 2º campo nunca passa de 12 (=> dia primeiro);
# com hífen o 1º campo nunca passa de 12 (=> mês primeiro).
# try_to_date devolve NULL (sem erro) quando nenhum formato serve (ex.: texto deslocado).
df_info = df_info.withColumn("data_lancamento", F.coalesce(
    F.expr("try_to_date(trim(data_lancamento), 'yyyy-MM-dd')"),
    F.expr("try_to_date(trim(data_lancamento), 'MM-dd-yyyy')"),
    F.expr("try_to_date(trim(data_lancamento), 'dd/MM/yyyy')"),
    F.expr("try_to_date(trim(data_lancamento), 'MM/dd/yyyy')")))

# Conversão segura da duração (texto deslocado como "en" vira NULL) + coluna derivada
df_info = df_info.withColumn("duracao_minutos", F.expr("try_cast(duracao_minutos AS INT)")) \
                 .withColumn("ano_lancamento", F.year("data_lancamento"))

# Deduplicação: 1 linha por filme (a origem tem ~9 mil linhas repetidas)
df_info = dedup(df_info, "id_filme")

salvar(df_info.select("id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
                      "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"),
       "tb_info_filmes")
display(spark.table(f"{S}.tb_info_filmes").groupBy("status_filme").count())

OK: tb_info_filmes (97611 linhas)


status_filme,count
Lançado,96261
Em Produção,604
Pós-Produção,697
Planejado,47
Não Informado,2


In [0]:
def limpa_moeda(c):
    """Higieniza valores monetários:
    - ausência textual (Unknown / Não Informado) -> NULL antes da conversão
    - remove símbolos e prefixos ("$ 97000000", "USD 150000000")
    - entende sufixo K/M/B ("34.0M" = 34.000.000), senão viraria US$ 34 sem ninguém perceber"""
    s = F.upper(F.trim(F.col(c)))
    s = F.when(F.lower(s).isin("unknown", "não informado", "nao informado", ""), None).otherwise(s)
    numero = F.regexp_extract(s, r"(-?\d+(?:\.\d+)?)", 1)
    sufixo = F.regexp_extract(s, r"([KMB])$", 1)
    mult = F.when(sufixo == "K", 1e3).when(sufixo == "M", 1e6).when(sufixo == "B", 1e9).otherwise(1.0)
    return F.when(numero != "", (numero.cast("double") * mult).cast("decimal(18,2)"))

df_fin = spark.table(f"{B}.tb_movies_financials").select(
    F.trim("id").alias("id_filme"),
    limpa_moeda("budget").alias("orcamento_usd"),
    limpa_moeda("revenue").alias("receita_usd"),
    "ingestion_datetime",
)

# Valores zerados ou negativos são tratados como ausentes (ex.: receitas negativas na origem)
df_fin = df_fin.withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd"))) \
               .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))

# Garante 1 linha por filme (a origem repete ids com valores diferentes)
df_fin = dedup(df_fin, "id_filme")

# Taxa de câmbio: cotação mais recente obtida da API do Banco Central
df_cotacao_atual = (spark.table(f"{B}.tb_cotacao_dolar")
                    .orderBy(F.col("dataHoraCotacao").desc()).limit(1)
                    .select(F.col("cotacaoCompra").cast("decimal(18,6)").alias("cotacao_dolar")))
df_fin = df_fin.crossJoin(df_cotacao_atual)

df_fin = (df_fin
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar")).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_dolar")).cast("decimal(18,2)"))
    # Lucro: se orçamento ou receita for NULL, o lucro fica NULL (não inventamos valor) e nada quebra
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    # Margem de lucro % = lucro / receita (só calcula se receita > 0: evita divisão por zero/NULL)
    .withColumn("margem_lucro_percentual",
                F.when(F.col("receita_usd") > 0, F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2))))

salvar(df_fin.select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
                     "lucro_usd", "lucro_brl", "margem_lucro_percentual"),
       "tb_financeiro_filmes")
display(spark.table(f"{S}.tb_financeiro_filmes").limit(10))

OK: tb_financeiro_filmes (99006 linhas)


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null
1000030,null,null,null,null,null,null,null
1000054,null,null,null,null,null,null,null
1000058,4700000.00,null,23466630.00,null,null,null,null
1000059,null,null,null,null,null,null,null
1000073,6000000.00,null,29957400.00,null,null,null,null


In [0]:
def inteiro_seguro(c):
    # Column shift: só aceita dígitos puros. "6.8" (nota que caiu na coluna de votos) ou texto -> NULL
    return F.when(F.trim(F.col(c)).rlike(r"^\d+$"), F.expr(f"try_cast(trim({c}) AS INT)"))

def decimal_seguro(c):
    return F.expr(f"try_cast(trim({c}) AS DOUBLE)")   # texto deslocado vira NULL sem quebrar o pipeline

# Assinatura do column shift: texto na coluna averageRating => a linha está deslocada e a "popularity"
# pode ser o valor de outra coluna (ex.: um ano como 2020 ou 1969). Nesses casos -> NULL
ar = F.trim(F.col("averageRating"))
deslocada = F.coalesce((ar != "") & (~ar.rlike(r"^-?\d+(\.\d+)?$")), F.lit(False))

df_met = spark.table(f"{B}.tb_movies_metrics").select(
    F.trim("id").alias("id_filme"),
    # Popularidade vem com vírgula decimal ("154,34"): padroniza para ponto ANTES do cast
    F.when(~deslocada,
           F.expr("try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE)")).alias("popularidade"),
    decimal_seguro("vote_average").alias("nota_media_tmdb"),
    inteiro_seguro("vote_count").alias("qtd_votos_tmdb"),
    decimal_seguro("averageRating").alias("nota_media_imdb"),
    inteiro_seguro("numVotes").alias("qtd_votos_imdb"),
    "ingestion_datetime",
)

# Regras de negócio: notas fora de 0-10 (inclui erro de escala, ex.: 65.5) -> NULL;
# popularidade e contagens de votos negativas -> NULL
df_met = (df_met
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb"))))

df_met = dedup(df_met, "id_filme")

salvar(df_met.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
                     "nota_media_imdb", "qtd_votos_imdb"), "tb_metricas_engajamento")
display(spark.table(f"{S}.tb_metricas_engajamento").orderBy(F.desc("popularidade")).limit(10))

OK: tb_metricas_engajamento (95115 linhas)


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
565770,2994.357,7.139,1023,5.9,119501
980489,2680.593,8.068,null,7.1,138696
968051,1692.778,6.545,365,5.6,84018
615656,1567.273,6.912,2034,null,105031
762430,1547.22,6.816,258,null,29152
1008042,1458.514,7.214,973,7.1,224148
385687,1175.267,null,3881,5.7,149785
678512,1111.036,7.973,503,7.6,null
346698,1069.34,7.279,5074,6.8,704472
976573,1008.942,7.757,2467,7.0,169047


In [0]:
df_av = spark.table(f"{B}.tb_movies_reviews").select(
    F.trim("id").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.expr("try_cast(trim(nota) AS DOUBLE)").alias("nota_usuario"),
    F.col("comentario").alias("comentario_usuario"),
)

# Nota válida: 0 a 10. Fora da faixa -> NULL
df_av = df_av.withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))

# Comentário nulo, vazio ou só com espaços -> texto padronizado
com = F.trim(F.col("comentario_usuario"))
df_av = df_av.withColumn("comentario_usuario",
                         F.when(com.isNull() | (com == ""), F.lit("Sem comentário")).otherwise(F.col("comentario_usuario")))

# Duplicados integrais (mesmo filme, usuário, nota e comentário)
df_av = df_av.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])

salvar(df_av, "tb_avaliacoes_usuarios")

OK: tb_avaliacoes_usuarios (32412 linhas)


In [0]:
# Separadores inconsistentes na origem: vírgula, ponto e vírgula e pipe ("Comedy|Drama")
df_gen = (spark.table(f"{B}.tb_credits_and_tags")
          .select(F.trim("id").alias("id_filme"), F.explode(F.split(F.col("genres"), r"[;|,]")).alias("nome_genero"))
          .withColumn("nome_genero", F.initcap(F.trim(F.col("nome_genero")))))

# Column shift: na coluna de gêneros caíram números ("0.6"), caminhos de imagem, frases de divulgação
# e até nomes de países. Filtro em 2 etapas:
# 1) só letras/espaço/hífen e curto -> remove vazios, numéricos, caminhos e frases com pontuação
df_gen = df_gen.filter(F.col("nome_genero").rlike(r"^[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ \-]{1,24}$")).dropDuplicates()
# 2) frequência mínima, contada DEPOIS de deduplicar (a Bronze tem cargas repetidas que inflariam a contagem):
#    gênero real aparece centenas de vezes; fragmentos de texto, poucas
freq = df_gen.groupBy("nome_genero").count().filter("count >= 100").select("nome_genero")
df_gen = df_gen.join(freq, "nome_genero").select("id_filme", "nome_genero")

salvar(df_gen, "tb_generos")
display(spark.table(f"{S}.tb_generos").groupBy("nome_genero").count().orderBy(F.desc("count")))

OK: tb_generos (141941 linhas)


nome_genero,count
Drama,32615
Documentary,19235
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
Tv Movie,4115


In [0]:
cred = spark.table(f"{B}.tb_credits_and_tags")

def entidades(coluna, tipo):
    return (cred.select(F.trim("id").alias("id_filme"),
                        F.explode(F.split(F.col(coluna), r"[;|,]")).alias("nome_entidade"))
                .withColumn("tipo_entidade", F.lit(tipo)))

# Mapeamento exigido: cast->Ator, directors->Diretor, writers->Roteirista, production_companies->Produtora
df_pe = (entidades("cast", "Ator").unionByName(entidades("directors", "Diretor"))
         .unionByName(entidades("writers", "Roteirista"))
         .unionByName(entidades("production_companies", "Produtora")))

# Padroniza capitalização (para "brad pitt" e "BRAD PITT" deduplicarem juntos)
df_pe = df_pe.withColumn("nome_entidade", F.initcap(F.trim(F.col("nome_entidade"))))

n = F.col("nome_entidade")
df_pe = df_pe.filter(
    n.isNotNull() & (n != "")
    & (~F.lower(n).isin("nenhum", "none", "unknown", "não informado", "n/a", "na", "null", "nan"))
    & (~n.rlike(r"^\d+(\.\d+)?$"))          # números deslocados
    & (F.length(n) <= 60)                   # textos descritivos longos (column shift)
    & (~n.rlike(r'[\\"?!]'))                # aspas/barras soltas e frases
    & (~n.rlike(r"^/"))                     # caminhos de imagem
).dropDuplicates()

salvar(df_pe, "tb_pessoas_empresas")
display(spark.table(f"{S}.tb_pessoas_empresas").groupBy("tipo_entidade").count())

OK: tb_pessoas_empresas (902159 linhas)


tipo_entidade,count
Ator,546139
Diretor,103588
Roteirista,133046
Produtora,119386


In [0]:
# Uma cotação por dia (último boletim do dia) e série CONTÍNUA com forward fill:
# fins de semana e feriados recebem o valor do último dia útil disponível
df_cot = (spark.table(f"{B}.tb_cotacao_dolar")
          .withColumn("moeda", F.lit("USD"))
          .withColumn("data_cotacao", F.to_date("dataHoraCotacao")))

janela_dia = Window.partitionBy("moeda", "data_cotacao").orderBy(F.col("dataHoraCotacao").desc())
df_cot = df_cot.withColumn("rn", F.row_number().over(janela_dia)).filter("rn = 1").drop("rn")

lim = df_cot.select(F.min("data_cotacao").alias("dmin"), F.max("data_cotacao").alias("dmax")).first()

if lim and lim["dmin"]:
    cal = (spark.range(1)
           .select(F.explode(F.sequence(F.lit(lim["dmin"]), F.lit(lim["dmax"]), F.expr("interval 1 day"))).alias("data_cotacao"))
           .withColumn("moeda", F.lit("USD")))

    df_silver_cot = cal.join(
        df_cot.select("moeda", "data_cotacao", F.col("cotacaoCompra").cast("double").alias("cotacao_compra")),
        on=["moeda", "data_cotacao"], how="left")

    w_ff = Window.partitionBy("moeda").orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
    df_silver_cot = df_silver_cot.withColumn("cotacao_compra",
                                             F.last("cotacao_compra", ignorenulls=True).over(w_ff))
    salvar(df_silver_cot, "tb_cotacao_dolar")

OK: tb_cotacao_dolar (8 linhas)


In [0]:
for t in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento",
          "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas", "tb_cotacao_dolar"]:
    df = spark.table(f"{S}.{t}")
    print(t, df.count())
    df.printSchema()

# Unicidade por filme: as duas contagens de cada tabela devem ser iguais
for t in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento"]:
    df = spark.table(f"{S}.{t}")
    print(t, df.count(), df.select("id_filme").distinct().count())

tb_info_filmes 97611
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)

tb_financeiro_filmes 99006
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(25,2) (nullable = true)

tb_metricas_engajamento 95115
root
 |-- id_filme: string (nullable = true)
 |-- popularidade: d